In [2]:
from google.colab import files
uploaded = files.upload()

Saving Messstationen Tagesdaten v2_20000101T0000_20231231T0000 (1).csv to Messstationen Tagesdaten v2_20000101T0000_20231231T0000 (1).csv


In [10]:
# T2.5 – Load Data into DBRepo
# Frost Day Prediction in Vienna
# Owner: C (Anxhela)

# ─────────────────────────────────────────────
# STEP 1: Install and import libraries
# ─────────────────────────────────────────────
!pip install dbrepo pandas --quiet

import pandas as pd
from dbrepo.RestClient import RestClient
from google.colab import files

# ─────────────────────────────────────────────
# STEP 2: Upload CSV
# ─────────────────────────────────────────────
uploaded = files.upload()

# ─────────────────────────────────────────────
# STEP 3: Connect to DBRepo
# ─────────────────────────────────────────────
client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username="DataStewGroup4",
    password="fakjyq-tastU6-pykcep"
)
print("Logged in as:", client.whoami())

# ─────────────────────────────────────────────
# STEP 4: Define IDs
# ─────────────────────────────────────────────
DATABASE_ID      = "a16a980a-3489-492b-adcf-74c70a07248b"
STATION_TABLE_ID = "9c57f7ff-99b6-454b-8d51-cff340ecf934"
OBS_TABLE_ID     = "21e39deb-1db9-4d34-bc04-45fa5cef72a0"

# ─────────────────────────────────────────────
# STEP 5: Load and clean the CSV
# ─────────────────────────────────────────────
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f"Rows loaded: {len(df)}")

# Convert date format
df["time"] = pd.to_datetime(df["time"]).dt.date

# Replace -1 (missing value indicator) with None
df = df.replace(-1, None)

print("After cleaning:")
print(df.head())

# ─────────────────────────────────────────────
# STEP 6: Prepare and load station table
# ─────────────────────────────────────────────
try:
    client.create_table_data(
        database_id=DATABASE_ID,
        table_id=STATION_TABLE_ID,
        data={
            "station_code": "5904",
            "name":         "Wien-Hohe Warte",
            "latitude":     48.24889,
            "longitude":    16.35639,
            "altitude_m":   202
        }
    )
    print("✅ Station row inserted")
except Exception as e:
    print(f"⚠️ Station insert skipped: {e}")
# ─────────────────────────────────────────────
# STEP 7: Prepare observations dataframe
# ─────────────────────────────────────────────
obs_df = pd.DataFrame({
    "station_id":       5904,
    "obs_date":         df["time"].astype(str),
    "temp_mean_c":      df["tl_mittel"],
    "temp_max_c":       df["tlmax"],
    "temp_min_c":       df["tlmin"],
    "precipitation_mm": df["rr"],
    "sunshine_h":       df["so_h"],
    "humidity_pct":     df["rf_mittel"],
    "visibility_m":     df["vv_mittel"]
})

print(f"\nObservations to insert: {len(obs_df)}")
print(obs_df.head())

# ─────────────────────────────────────────────
# STEP 8: Load observations into DBRepo
# ─────────────────────────────────────────────
success = 0
errors = 0

for _, row in obs_df.iterrows():
    try:
        client.create_table_data(
            database_id=DATABASE_ID,
            table_id=OBS_TABLE_ID,
            data={
                "station_id":       int(row["station_id"]),
                "obs_date":         str(row["obs_date"]),
                "temp_mean_c":      None if pd.isna(row["temp_mean_c"]) else float(row["temp_mean_c"]),
                "temp_max_c":       None if pd.isna(row["temp_max_c"]) else float(row["temp_max_c"]),
                "temp_min_c":       None if pd.isna(row["temp_min_c"]) else float(row["temp_min_c"]),
                "precipitation_mm": None if pd.isna(row["precipitation_mm"]) else float(row["precipitation_mm"]),
                "sunshine_h":       None if pd.isna(row["sunshine_h"]) else float(row["sunshine_h"]),
                "humidity_pct":     None if pd.isna(row["humidity_pct"]) else float(row["humidity_pct"]),
                "visibility_m":     None if pd.isna(row["visibility_m"]) else float(row["visibility_m"])
            }
        )
        success += 1
        if success % 100 == 0:
            print(f"Inserted {success} rows...")
    except Exception as e:
        errors += 1
        if errors <= 3:
            print(f"❌ Row error: {e}")

print(f"\n✅ Done! {success} rows inserted, {errors} errors")

Saving Messstationen Tagesdaten v2_20000101T0000_20231231T0000 (1).csv to Messstationen Tagesdaten v2_20000101T0000_20231231T0000 (1) (3).csv
DataStewGroup4
Logged in as: DataStewGroup4
Rows loaded: 8766
After cleaning:
         time  station tl_mittel tlmax tlmin    rr  so_h  rf_mittel  vv_mittel
0  2000-01-01     5904      -1.5   0.4  -3.4   7.3   0.0         88        1.0
1  2000-01-02     5904      -1.6  -0.5  -2.6   1.6   0.0         96        1.0
2  2000-01-03     5904      -1.6   1.6  -4.8  None   2.7         97        1.0
3  2000-01-04     5904      -0.6   3.9  -5.0   0.4   4.0         76        5.2
4  2000-01-05     5904       4.5   8.8   0.1  None   2.8         78        3.2
⚠️ Station insert skipped: Failed to insert table data: {"status":"BAD_REQUEST","message":"Failed to create tuple: (conn=46372) Field 'station_id' doesn't have a default value","code":"error.query.invalid"}

Observations to insert: 8766
   station_id    obs_date temp_mean_c temp_max_c temp_min_c precipita

KeyboardInterrupt: 

In [11]:
# Fix station insert with explicit station_id
try:
    client.create_table_data(
        database_id=DATABASE_ID,
        table_id=STATION_TABLE_ID,
        data={
            "station_id":   5904,
            "station_code": "5904",
            "name":         "Wien-Hohe Warte",
            "latitude":     48.24889,
            "longitude":    16.35639,
            "altitude_m":   202
        }
    )
    print("✅ Station row inserted")
except Exception as e:
    print(f"⚠️ Station result: {e}")

✅ Station row inserted


In [12]:
# STEP 8: Load observations into DBRepo
success = 0
errors = 0

for _, row in obs_df.iterrows():
    try:
        client.create_table_data(
            database_id=DATABASE_ID,
            table_id=OBS_TABLE_ID,
            data={
                "station_id":       int(row["station_id"]),
                "obs_date":         str(row["obs_date"]),
                "temp_mean_c":      None if pd.isna(row["temp_mean_c"]) else float(row["temp_mean_c"]),
                "temp_max_c":       None if pd.isna(row["temp_max_c"]) else float(row["temp_max_c"]),
                "temp_min_c":       None if pd.isna(row["temp_min_c"]) else float(row["temp_min_c"]),
                "precipitation_mm": None if pd.isna(row["precipitation_mm"]) else float(row["precipitation_mm"]),
                "sunshine_h":       None if pd.isna(row["sunshine_h"]) else float(row["sunshine_h"]),
                "humidity_pct":     None if pd.isna(row["humidity_pct"]) else float(row["humidity_pct"]),
                "visibility_m":     None if pd.isna(row["visibility_m"]) else float(row["visibility_m"])
            }
        )
        success += 1
        if success % 100 == 0:
            print(f"Inserted {success} rows...")
    except Exception as e:
        errors += 1
        if errors <= 3:
            print(f"❌ Row error: {e}")

print(f"\n✅ Done! {success} rows inserted, {errors} errors")

Inserted 100 rows...
Inserted 200 rows...
Inserted 300 rows...
Inserted 400 rows...
Inserted 500 rows...
Inserted 600 rows...
Inserted 700 rows...
Inserted 800 rows...
Inserted 900 rows...
Inserted 1000 rows...
Inserted 1100 rows...
Inserted 1200 rows...
Inserted 1300 rows...
Inserted 1400 rows...
Inserted 1500 rows...
Inserted 1600 rows...
Inserted 1700 rows...
Inserted 1800 rows...
Inserted 1900 rows...
Inserted 2000 rows...
Inserted 2100 rows...
Inserted 2200 rows...
Inserted 2300 rows...
Inserted 2400 rows...
Inserted 2500 rows...
Inserted 2600 rows...
Inserted 2700 rows...
Inserted 2800 rows...
Inserted 2900 rows...
Inserted 3000 rows...
Inserted 3100 rows...
Inserted 3200 rows...
Inserted 3300 rows...
Inserted 3400 rows...
Inserted 3500 rows...
Inserted 3600 rows...
Inserted 3700 rows...
Inserted 3800 rows...
Inserted 3900 rows...
Inserted 4000 rows...
Inserted 4100 rows...
Inserted 4200 rows...
Inserted 4300 rows...
Inserted 4400 rows...
Inserted 4500 rows...
Inserted 4600 rows.

In [13]:
# Check which rows failed
errors_list = []

for _, row in obs_df.iterrows():
    try:
        client.create_table_data(
            database_id=DATABASE_ID,
            table_id=OBS_TABLE_ID,
            data={
                "station_id":       int(row["station_id"]),
                "obs_date":         str(row["obs_date"]),
                "temp_mean_c":      None if pd.isna(row["temp_mean_c"]) else float(row["temp_mean_c"]),
                "temp_max_c":       None if pd.isna(row["temp_max_c"]) else float(row["temp_max_c"]),
                "temp_min_c":       None if pd.isna(row["temp_min_c"]) else float(row["temp_min_c"]),
                "precipitation_mm": None if pd.isna(row["precipitation_mm"]) else float(row["precipitation_mm"]),
                "sunshine_h":       None if pd.isna(row["sunshine_h"]) else float(row["sunshine_h"]),
                "humidity_pct":     None if pd.isna(row["humidity_pct"]) else float(row["humidity_pct"]),
                "visibility_m":     None if pd.isna(row["visibility_m"]) else float(row["visibility_m"])
            }
        )
    except Exception as e:
        errors_list.append({
            "date": str(row["obs_date"]),
            "error": str(e)
        })

print(f"Failed rows: {len(errors_list)}")
for err in errors_list[:5]:
    print(f"Date: {err['date']} | Error: {err['error']}")

Failed rows: 8744
Date: 2000-01-01 | Error: Failed to insert table data: {"status":"BAD_REQUEST","message":"Failed to create tuple: (conn=61544) Duplicate entry '2000-01-01-5904' for key 'PRIMARY'","code":"error.query.invalid"}
Date: 2000-01-02 | Error: Failed to insert table data: {"status":"BAD_REQUEST","message":"Failed to create tuple: (conn=15142) Duplicate entry '2000-01-02-5904' for key 'PRIMARY'","code":"error.query.invalid"}
Date: 2000-01-03 | Error: Failed to insert table data: {"status":"BAD_REQUEST","message":"Failed to create tuple: (conn=15313) Duplicate entry '2000-01-03-5904' for key 'PRIMARY'","code":"error.query.invalid"}
Date: 2000-01-04 | Error: Failed to insert table data: {"status":"BAD_REQUEST","message":"Failed to create tuple: (conn=61548) Duplicate entry '2000-01-04-5904' for key 'PRIMARY'","code":"error.query.invalid"}
Date: 2000-01-05 | Error: Failed to insert table data: {"status":"BAD_REQUEST","message":"Failed to create tuple: (conn=15318) Duplicate entry

In [1]:
# T2.5 - Verify views return correct results

VIEW_NAMES = [
    "vw_monthly_frost_summary",
    "vw_ml_complete_cases",
    "vw_ml_daily_features"
]

for view_name in VIEW_NAMES:
    try:
        result = client.get_view_data(
            database_id=DATABASE_ID,
            view_name=view_name
        )
        print(f"✅ {view_name} — rows returned: {len(result)}")
    except Exception as e:
        print(f"❌ {view_name} — error: {e}")

❌ vw_monthly_frost_summary — error: name 'client' is not defined
❌ vw_ml_complete_cases — error: name 'client' is not defined
❌ vw_ml_daily_features — error: name 'client' is not defined


In [2]:
!pip install dbrepo --quiet
from dbrepo.RestClient import RestClient

client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username="DataStewGroup4",
    password="fakjyq-tastU6-pykcep"
)

DATABASE_ID = "a16a980a-3489-492b-adcf-74c70a07248b"

print("Logged in as:", client.whoami())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.0/165.0 kB 9.7 MB/s eta 0:00:00
DataStewGroup4
Logged in as: DataStewGroup4


In [4]:
# Check get_view_data signature
import inspect
print(inspect.signature(client.get_view_data))

(database_id: str, view_id: str, page: int = 0, size: int = 1000000) -> pandas.core.frame.DataFrame


In [5]:
# Get all views and their IDs
views = client.get_views(database_id=DATABASE_ID)
for v in views:
    print(f"View: {v.name} | ID: {v.id}")

View: vw_monthly_frost_summary | ID: 5cb817ff-670d-4142-a1d8-c43cce6997bf
View: vw_ml_complete_cases | ID: f868d4e8-5505-4ec8-9a9b-c2de34e2c545
View: vw_ml_daily_features | ID: 59cafa7d-0267-4c6a-a94d-6bf4670badf7


In [6]:
# T2.5 - Verify views return correct results
views_map = {
    "vw_monthly_frost_summary": "5cb817ff-670d-4142-a1d8-c43cce6997bf",
    "vw_ml_complete_cases":     "f868d4e8-5505-4ec8-9a9b-c2de34e2c545",
    "vw_ml_daily_features":     "59cafa7d-0267-4c6a-a94d-6bf4670badf7"
}

for view_name, view_id in views_map.items():
    try:
        result = client.get_view_data(
            database_id=DATABASE_ID,
            view_id=view_id
        )
        print(f"✅ {view_name} — rows returned: {len(result)}")
        print(result.head(3))
        print()
    except Exception as e:
        print(f"❌ {view_name} — error: {e}")

✅ vw_monthly_frost_summary — rows returned: 8766
     obs_date  precipitation_mm  temp_min_c
0  2002-12-05               2.5         2.8
1  2004-08-18               NaN        17.4
2  2007-06-16               1.1        15.8

✅ vw_ml_complete_cases — rows returned: 8766
   humidity_pct    obs_date  precipitation_mm  sunshine_h  temp_max_c  \
0          75.0  2022-04-05               2.7         0.3        10.1   
1          71.0  2014-08-22               0.0         5.3        22.8   
2          82.0  2011-02-19               0.0         0.0         3.5   

   temp_mean_c  temp_min_c  visibility_m  
0          7.4         4.6           6.7  
1         17.7        12.5           3.8  
2          3.0         2.5           2.6  

✅ vw_ml_daily_features — rows returned: 8766
   humidity_pct    obs_date  precipitation_mm  sunshine_h  temp_max_c  \
0          75.0  2022-04-05               2.7         0.3        10.1   
1          71.0  2014-08-22               0.0         5.3        22.8   